In [ ]:
import mne

In [29]:
class SleepEDFPreprocessingPipeline:
    def __init__(self, file_path, l_freq=0.3, h_freq=40., notch_freq=50):
        self.file_path = file_path
        self.l_freq = l_freq
        self.h_freq = h_freq
        self.notch_freq = notch_freq
        self.raw = None
        self.epochs = None

    def load_data(self):
        self.raw = mne.io.read_raw_edf(self.file_path, preload=True)
        print(f"Loaded data: {self.raw.info['nchan']} channels, {self.raw.info['sfreq']} Hz")
        return self

    def select_eeg_channels(self):
        eeg_channels = ['EEG Fpz-Cz', 'EEG Pz-Oz']
        self.raw.pick_channels(eeg_channels)
        print(f"Selected EEG channels: {eeg_channels}")
        return self

    def set_montage(self):
        # Rename channels to match montage names (approximation)
        rename_map = {'EEG Fpz-Cz': 'Fpz', 'EEG Pz-Oz': 'Pz'}
        self.raw.rename_channels(rename_map)
        montage = mne.channels.make_standard_montage('standard_1020')
        self.raw.set_montage(montage, on_missing='ignore')
        print("Montage set (approximate for bipolar channels).")
        return self

    def filter_data(self):
        sfreq = self.raw.info['sfreq']
        nyquist = sfreq / 2
        safe_margin = 0.5 #?
        if self.h_freq is not None and self.h_freq >= nyquist - safe_margin:
            self.h_freq = nyquist - safe_margin
            print(f"Adjusted h_freq to {self.h_freq} Hz (safe margin applied).")
        self.raw.filter(l_freq=self.l_freq, h_freq=self.h_freq, fir_design='firwin')
        #self.raw.notch_filter(freqs=[self.notch_freq])
        print(f"Applied bandpass {self.l_freq}-{self.h_freq} Hz and notch {self.notch_freq} Hz.")
        return self
    
    def re_reference(self):
        self.raw.set_eeg_reference('average')
        print("Re-referenced to average.")
        return self

    def apply_ica(self, n_components=15):
        n_channels = len(self.raw.ch_names)
        ica = mne.preprocessing.ICA(n_components=n_channels, random_state=97)
        ica.fit(self.raw)
        
        if 'eog' in self.raw.get_channel_types():
            eog_indices, eog_scores = ica.find_bads_eog(self.raw)
            ica.exclude = eog_indices
        else:
            print("No EOG channels found. Skipping EOG artifact detection.")
        self.raw = ica.apply(self.raw)
        print(f"ICA applied. Excluded components: {ica.exclude}")
        return self

    def save_data(self, out_path):
        self.raw.save(out_path, overwrite=True)
        print(f"Saved preprocessed data to {out_path}")
        return self

In [31]:
pipeline = SleepEDFPreprocessingPipeline('sleepdata/SC4001E0-PSG.edf')
pipeline.load_data()\
        .select_eeg_channels()\
        .set_montage()\
        .filter_data()\
        .re_reference()\
        .save_data('cleaned_sleepedf_raw.fif')

Extracting EDF parameters from C:\Users\onontsatsalg\Documents\Code\PersonalAI\sleepdata\SC4001E0-PSG.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 7949999  =      0.000 ... 79499.990 secs...


C:\Users\onontsatsalg\AppData\Local\Temp\ipykernel_836\2556003947.py:13: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  self.raw = mne.io.read_raw_edf(self.file_path, preload=True)
C:\Users\onontsatsalg\AppData\Local\Temp\ipykernel_836\2556003947.py:13: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  self.raw = mne.io.read_raw_edf(self.file_path, preload=True)
C:\Users\onontsatsalg\AppData\Local\Temp\ipykernel_836\2556003947.py:13: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  self.raw = mne.io.read_raw_edf(self.file_path, preload=True)


Loaded data: 7 channels, 100.0 Hz
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Selected EEG channels: ['EEG Fpz-Cz', 'EEG Pz-Oz']
Montage set (approximate for bipolar channels).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.3 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.30
- Lower transition bandwidth: 0.30 Hz (-6 dB cutoff frequency: 0.15 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1101 samples (11.010 s)

Applied bandpass 0.3-40.0 Hz and notch 50 Hz.
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Re-referenced to average.
Writing C:\Users\onontsatsalg\Documents

In [ ]:
### Pipeline

In [ ]:
import mne

class EEGPreprocessingPipeline:
    def __init__(self, file_path, montage='standard_1020', l_freq=1., h_freq=40., notch_freq=50):
        self.file_path = file_path
        self.montage = montage
        self.l_freq = l_freq
        self.h_freq = h_freq
        self.notch_freq = notch_freq
        self.raw = None

    def load_data(self):
        self.raw = mne.io.read_raw_edf(self.file_path, preload=True)
        print("Data loaded.")
        return self

    def set_montage(self):
        self.raw.set_montage(mne.channels.make_standard_montage(self.montage))
        print("Montage set.")
        return self

    def filter_data(self):
        self.raw.filter(l_freq=self.l_freq, h_freq=self.h_freq)
        self.raw.notch_filter(freqs=[self.notch_freq])
        print("Filtering done.")
        return self

    def apply_ica(self, n_components=15):
        ica = mne.preprocessing.ICA(n_components=n_components, random_state=97)
        ica.fit(self.raw)
        # You can add logic to detect and exclude blink components
        self.raw = ica.apply(self.raw)
        print("ICA applied.")
        return self

    def save_data(self, out_path):
        self.raw.save(out_path, overwrite=True)
        print(f"Data saved to {out_path}.")
        return self

In [ ]:
pipeline = EEGPreprocessingPipeline('your_file.edf')
pipeline.load_data().set_montage().filter_data().apply_ica().save_data('cleaned_raw.fif')

In [ ]:
#Pipeline first draft

In [1]:
import mne

In [2]:
# 1. Load raw EEG data
raw = mne.io.read_raw_edf('sleepdata/SC4001E0-PSG.edf', preload=True)

Extracting EDF parameters from C:\Users\onontsatsalg\Documents\Code\PersonalAI\sleepdata\SC4001E0-PSG.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...


C:\Users\onontsatsalg\AppData\Local\Temp\ipykernel_836\3320410700.py:2: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf('sleepdata/SC4001E0-PSG.edf', preload=True)
C:\Users\onontsatsalg\AppData\Local\Temp\ipykernel_836\3320410700.py:2: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf('sleepdata/SC4001E0-PSG.edf', preload=True)
C:\Users\onontsatsalg\AppData\Local\Temp\ipykernel_836\3320410700.py:2: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf('sleepdata/SC4001E0-PSG.edf', preload=True)


Reading 0 ... 7949999  =      0.000 ... 79499.990 secs...


In [3]:
# 2. Set montage
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

ValueError: DigMontage is only a subset of info. There are 7 channel positions not present in the DigMontage. The channels missing from the montage are:

['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker'].

Consider using inst.rename_channels to match the montage nomenclature, or inst.set_channel_types if these are not EEG channels, or use the on_missing parameter if the channel positions are allowed to be unknown in your analyses.

In [ ]:
# 3. Filter
raw.filter(l_freq=1., h_freq=40.)
raw.notch_filter(freqs=[50])  # For line noise

In [ ]:
# 4. Re-reference
raw.set_eeg_reference('average')

In [ ]:
# 5. Artifact removal with ICA
ica = mne.preprocessing.ICA(n_components=15, random_state=97)
ica.fit(raw)
ica.exclude = [0]  # Example: exclude component for eye blink
raw = ica.apply(raw)

In [ ]:
# 6. Epoching
events = mne.find_events(raw)
event_id = {'stimulus': 1}
epochs = mne.Epochs(raw, events, event_id, tmin=-0.2, tmax=0.8, baseline=(None, 0), preload=True)

In [ ]:
# 7. Save
epochs.save('preprocessed-epo.fif', overwrite=True)